# Notebook 1 — Markov Decision Processes, Value Functions and Policies

<div class="alert alert-success">

**Learning outcomes:**
By the end of this notebook you should be able to:
- state the Markov property and the MDP formalism,
- define trajectories, policies (deterministic and stochastic),
- define the discounted return, the value function $v^\pi$ and the action-value function $q^\pi$,
- explain what an optimal policy is and why a deterministic one always exists,
- frame the V2G problem as an MDP.
</div>

# 1. In plain words

<div class="alert alert-success">

**Reinforcement Learning** is about learning an optimal sequential behaviour for a given dynamical system.
</div>

Breaking this down:
- **Dynamical system** → state variables that evolve through time under the influence of decision variables.
- **Sequential behaviour** → discrete time steps, a sequence of decisions.
- **Optimal** → a reward signal quantifies the quality of each transition.
- **Learning** → no known model a priori; behaviour is adapted from interaction samples.

Inspirations for RL:
- Control theory and stochastic processes → **modelling**
- Statistics, optimisation, cognitive psychology → **learning**

# 2. Formalising the ingredients

## 2.1 States

At each time step $t$ the system is described by a **state** $S_t$, a random variable taking values in a state space $\mathcal{S}$.

## 2.2 Actions

The decision maker chooses an **action** $A_t$ from an action space $\mathcal{A}$.

## 2.3 Rewards

After applying $A_t$ in state $S_t$ the environment transitions to $S_{t+1}$ and emits a scalar **reward** $R_t$.

The reward encodes our goal: a well-designed reward function guides the agent towards desirable behaviour.

## 2.4 Trajectories

A **trajectory** is a sequence of the form
$$\tau = (S_0, A_0, R_0, S_1, A_1, R_1, \ldots)$$
It tells us the full history of states, actions and rewards.

# 3. Markov Decision Processes

We make one fundamental assumption:

<div class="alert alert-success">

**Markov property**
$$\mathbb{P}(S_{t+1}, R_t \mid S_t, A_t, S_{t-1}, A_{t-1}, \ldots, S_0, A_0)
= \mathbb{P}(S_{t+1}, R_t \mid S_t, A_t).$$

The future is independent of the past *given the present state and action*.
</div>

A system satisfying the Markov property is called a **Markov Decision Process (MDP)**.

<div class="alert alert-success">

**MDP** — a 4-tuple $(\mathcal{S}, \mathcal{A}, p, r)$ where:
- $\mathcal{S}$: state space,
- $\mathcal{A}$: action space,
- $p(s' \mid s, a) = \mathbb{P}(S_{t+1}=s' \mid S_t=s, A_t=a)$: transition model,
- $r(s, a) = \mathbb{E}[R_t \mid S_t=s, A_t=a]$: expected reward function.
</div>

We will work mostly with **stationary, infinite-horizon** MDPs — the dynamics and rewards do not change over time and we care about the long run.

Let's load FrozenLake as our first illustrative example.

FrozenLake is a 4×4 grid where the agent must reach the goal (G) without falling into holes (H).
The ice is slippery: actions succeed with some probability and deviate randomly otherwise.

In [ ]:
import gymnasium as gym
import gymnasium.envs.toy_text.frozen_lake as fl
import numpy as np

env = gym.make('FrozenLake-v1', render_mode="ansi")
env.reset()
print(env.render())

print("State space :", env.observation_space)
print("Action space:", env.action_space)
print("Actions: LEFT=0, DOWN=1, RIGHT=2, UP=3")

In [ ]:
# Inspect one transition: state 1 (top row, 2nd cell), action RIGHT
state = 1
action = fl.RIGHT
outcomes = env.unwrapped.P[state][action]
print(f"Transitions from state {state} with action RIGHT:")
for prob, next_state, reward, done in outcomes:
    print(f"  -> s'={next_state}, r={reward}, done={done}, prob={prob:.2f}")

# 4. Policies

How do we specify a behaviour?

<div class="alert alert-success">

A **policy** $\pi$ is a mapping from states to a distribution over actions:
$$\pi : \mathcal{S} \to \Delta_{\mathcal{A}}$$
where $\Delta_{\mathcal{A}}$ is the set of probability distributions over $\mathcal{A}$.

- **Deterministic policy**: $\pi(s) \in \mathcal{A}$ — maps each state to a single action.
- **Stochastic policy**: $\pi(a \mid s) \in [0,1]$ — maps each state to a distribution over actions.
</div>

Key result (proved later):

<div class="alert alert-success">

**Theorem**: For any infinite-horizon, $\gamma$-discounted MDP there exists at least one optimal policy that is **stationary, deterministic and memoryless** (Markovian).
</div>

This justifies restricting our search to $\pi : \mathcal{S} \to \mathcal{A}$.

In [ ]:
# Three simple policies on FrozenLake
always_right = fl.RIGHT * np.ones(env.observation_space.n, dtype=int)
always_down  = fl.DOWN  * np.ones(env.observation_space.n, dtype=int)
random_pi    = np.random.randint(0, env.action_space.n, env.observation_space.n)

def run_episode(env, policy, horizon=200):
    """Run one episode; return total undiscounted reward."""
    total_r = 0.0
    state, _ = env.reset()
    for _ in range(horizon):
        action = int(policy[state])
        state, r, done, trunc, _ = env.step(action)
        total_r += r
        if done or trunc:
            break
    return total_r

# Estimate policy values by Monte Carlo
N = 10_000
right_returns = [run_episode(env, always_right) for _ in range(N)]
down_returns  = [run_episode(env, always_down)  for _ in range(N)]
rand_returns  = [run_episode(env, random_pi)    for _ in range(N)]

print(f"Always RIGHT : mean={np.mean(right_returns):.4f}, std={np.std(right_returns):.4f}")
print(f"Always DOWN  : mean={np.mean(down_returns):.4f},  std={np.std(down_returns):.4f}")
print(f"Random       : mean={np.mean(rand_returns):.4f},  std={np.std(rand_returns):.4f}")

# 5. Value Functions

To compare policies we need a scalar measure of their quality.

## 5.1 Discounted return

Given a trajectory starting in state $s$ and following policy $\pi$, the **$\gamma$-discounted return** is:
$$G^\pi(s) = \sum_{t=0}^{\infty} \gamma^t R_t \quad \Bigg| \quad S_0 = s,\ A_t \sim \pi(\cdot | S_t),\ S_{t+1} \sim p(\cdot | S_t, A_t).$$

The discount factor $\gamma \in [0,1)$ makes the sum finite and encodes how much the agent values future rewards relative to immediate ones.

## 5.2 State-value function

<div class="alert alert-success">

$$v^\pi(s) = \mathbb{E}\left[ G^\pi(s) \right] = \mathbb{E}\left[ \sum_{t=0}^\infty \gamma^t R_t \,\bigg|\, S_0 = s,\ \pi \right]$$

$v^\pi(s)$ is the **expected discounted return** from state $s$ under policy $\pi$.
</div>

## 5.3 Action-value function (Q-function)

<div class="alert alert-success">

$$q^\pi(s,a) = \mathbb{E}\left[ \sum_{t=0}^\infty \gamma^t R_t \,\bigg|\, S_0=s,\ A_0=a,\ A_{t\geq 1} \sim \pi \right]$$

$q^\pi(s,a)$ is the expected return when taking action $a$ in $s$ and *then* following $\pi$.
</div>

Note: $v^\pi(s) = \sum_a \pi(a|s)\, q^\pi(s,a)$ for stochastic $\pi$, or $v^\pi(s) = q^\pi(s, \pi(s))$ for deterministic $\pi$.

In [ ]:
def mc_value_estimation(env, policy, gamma=0.9, n_episodes=20_000, horizon=200):
    """Estimate v^pi for each state via Monte Carlo rollouts."""
    returns_sum = np.zeros(env.observation_space.n)
    counts      = np.zeros(env.observation_space.n)
    for _ in range(n_episodes):
        state, _ = env.reset()
        episode = []
        for _ in range(horizon):
            action = int(policy[state])
            next_state, r, done, trunc, _ = env.step(action)
            episode.append((state, r))
            state = next_state
            if done or trunc:
                break
        # Compute discounted returns
        G = 0.0
        visited = set()
        for s, r in reversed(episode):
            G = r + gamma * G
            if s not in visited:          # first-visit MC
                returns_sum[s] += G
                counts[s] += 1
                visited.add(s)
    V = np.where(counts > 0, returns_sum / counts, 0.0)
    return V

gamma = 0.99
V_right = mc_value_estimation(env, always_right, gamma=gamma)
V_down  = mc_value_estimation(env, always_down,  gamma=gamma)

print("Value of always-RIGHT (γ=0.99):")
print(V_right.reshape(4,4).round(3))
print()
print("Value of always-DOWN (γ=0.99):")
print(V_down.reshape(4,4).round(3))

# 6. Optimal Policies

<div class="alert alert-success">

A policy $\pi^*$ is **optimal** if it dominates every other policy in every state:
$$\forall s \in \mathcal{S},\ \forall \pi :\ v^{\pi^*}(s) \geq v^\pi(s).$$

All optimal policies share the same **optimal value function** $v^* = v^{\pi^*}$.
</div>

The optimal Q-function is:
$$q^*(s,a) = \max_\pi q^\pi(s,a)$$

An optimal policy can always be recovered greedily:
$$\pi^*(s) = \arg\max_a q^*(s,a).$$

This is the key link between **value functions** and **optimal policies**: find $q^*$, extract $\pi^*$ for free.

<div class="alert alert-warning">

**Exercise:**
Suppose you know $q^*$ for FrozenLake. Write a function `greedy_policy(Q, env)` that extracts the greedy policy as an array of actions.
</div>

In [ ]:
def greedy_policy(Q, env):
    """Q has shape (n_states, n_actions). Returns deterministic policy array."""
    return np.argmax(Q, axis=1)

# Placeholder — we will compute q* properly in Notebook 2.
# For now, let's just verify the function works with a random Q.
Q_random = np.random.randn(env.observation_space.n, env.action_space.n)
pi_greedy = greedy_policy(Q_random, env)
print("Greedy policy from random Q:", pi_greedy)

# 7. The Cartography of RL Methods

We now have two equivalent goals:
1. **Value optimisation**: find $v^*$ or $q^*$, then extract $\pi^*$ greedily. → Dynamic programming, Q-learning, DQN, ...
2. **Policy optimisation**: directly optimise $\pi_\theta$ with gradient ascent on $J(\pi) = \mathbb{E}_{s_0}[v^\pi(s_0)]$. → REINFORCE, A2C, PPO, ...

Both families will be covered in this class.

# 8. Back to the V2G problem

<div class="alert alert-warning">

**Discussion**
Revisit the ev2gym environment from Notebook 0 and answer:
1. Is the V2G problem a Markov Decision Process? Justify by stating what $\mathcal{S}$, $\mathcal{A}$, $p$ and $r$ are.
2. Is the process stationary? What is the horizon?
3. Is the state fully observable?
4. What makes the action space particularly challenging compared to FrozenLake?
</div>

In [ ]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.rl_agent.state import V2G_profit_max
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible

env_v2g = EV2Gym(config_file="custom.yaml", save_replay=False,
                 save_plots=False, state_function=V2G_profit_max)
state, _ = env_v2g.reset()

print("State space  :", env_v2g.observation_space)
print("Action space :", env_v2g.action_space)
print()
print("State dim:", env_v2g.observation_space.shape[0])
print("Action dim:", env_v2g.action_space.shape[0])
print()
print("State interpretation:")
print("  Continuous — prices, SoCs, remaining times, power usage")
print("Action interpretation:")
print("  Continuous in [-1, 1] — normalised charging/discharging current per CS")

## Summary

| Concept | FrozenLake | V2G |
|---|---|---|
| State space | Discrete, 16 states | Continuous, ~42 dim |
| Action space | Discrete, 4 actions | Continuous, 10-dim |
| Horizon | Finite (episodic) | Finite (80 steps) |
| Stochastic transitions | Yes (slippery ice) | Yes (EV arrivals, prices) |
| Fully observable | Yes | Approximately yes |

In the next notebook we will learn how to *compute* optimal value functions via Bellman equations and value iteration, and how to approximate them with neural networks (DQN).